# GENIE tune-weight mismatch between WC `weight_cv` and the Pandora `weights` map

**Summary.** Every MC event in our ntuples carries the MicroBooNE GENIE tune weight twice:

* **`wcpselection/T_eval : weight_cv`** (and `weight_spline`) — written by Wire-Cell. This is what `postprocessing.py`
  folds into every `wc_net_weight_*` column (`weight_cv * weight_spline`), and what `systematics.py` divides the stored
  universes by.
* **`nuselection/NeutrinoSelectionFilter : weights`** (the map of GENIE / flux / reint universes) plus the scalars
  `weightTune` / `weightSpline` — written by the Pandora ntuple code. `create_rw_syst_df.py` reads the universe lists
  from this map into `spline_weights_df.parquet` / `presel_weights_df.parquet`. Universe 0 of every tune-scaled knob
  (and the single entry of `TunedCentralValue_UBGenie`) *is* `weightTune`, because the stored GENIE universes are
  absolute weights `tune x knob_ratio`.

For an event with **one** GENIE interaction the two agree exactly. For an event with **several** interactions (pile-up
in one spill — the norm for dirt, rare for `nu_overlay`) they disagree ~60% of the time. Every truth branch in every tree
(WC `truth_nu*`, Pandora `nu_e`, gLEE `mctruth_*`, the eventweight tree, LANTERN) describes the *same* neutrino, and
**Pandora's `weightTune` is consistent with that neutrino's physics while WC's `weight_cv` is not**: in multi-interaction
events WC's `weight_cv` is non-unity for ~45% of NC / RES / DIS events (where the tune cannot act) and unity for ~55% of
CC QE / MEC events (where it must act). So it is WC's `weight_cv` (and, in Runs 1-3, its `weight_spline`) that is taken
from a different `MCTruth` than the neutrino WC itself reports. This hits ~49% of dirt events, ~0.4% of `nu_overlay`,
~0.01% of `nue_overlay`, and none of the dedicated pi0 samples.

Consequences:

1. **Our CV prediction** uses the wrong tune (and Runs 1-3 spline) weight for those events. Dirt is the sample that
   matters: about half of its events carry another interaction's weight.
2. **Our systematics** (`systematics.py`) divide every universe by `wc_weight_cv`; for those events the division is by
   the wrong number, so every universe is scaled by the same wrong constant.
3. **PROfit** (`force_0_cv="true"`) normalizes by universe 0 **per bin**, not per event. An event that should carry a
   tune weight of 4.5e-6 but was written with WC's 1.38 sits alone in the `multipi0_dirt` 700-800 MeV bin and turns the
   `XSecShape_CCMEC` knob (whose MEC variation universe is not tune-scaled) into a bin ratio of
   1.66 / 4.5e-6 = 3.7e5: the ~4000 fractional "MEC" error in PROfit's `*_fractional_systematics.pdf` and the
   265 events/MeV band in the `--legacy-postfit-error` plot (2026-09-09, `test_r15_21`). The spike is a symptom of the
   wrong CV weight.

This notebook demonstrates each step with the actual dataframes and ntuples (state of 2026-09-10).


In [ ]:
import sys, os, re
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import polars as pl
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 110

from src.file_locations import intermediate_files_location as IL, data_files_location as DL

CACHE_DIR = "genie_tune_mismatch_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# RSE alone is not unique across filetypes (data/ext/overlays reuse run/subrun/event), so joins use filename too
JOIN_KEYS = ["filename", "run", "subrun", "event"]

# fixed colors per filetype for every plot in this notebook (identity never changes with the series count)
FILETYPE_COLORS = {
    "nu_overlay": "#1f77b4", "nue_overlay": "#2ca02c", "dirt_overlay": "#8c564b",
    "nc_pi0_overlay": "#d62728", "numucc_pi0_overlay": "#9467bd",
}
FILETYPE_ORDER = list(FILETYPE_COLORS)
WC_COLOR, PANDORA_COLOR = "#e69f00", "#0072b2"

MISMATCH_TOL = 0.05   # |weightTune / weight_cv - 1| above this counts as a mismatch
SCAT_NAMES = {1: "QE", 3: "RES", 4: "DIS", 5: "COH", 7: "nu-e", 10: "MEC"}


## 1. Compare the two tune weights for every event with GENIE universes

`spline_weights_df.parquet` holds one row per preselected overlay event that has a GENIE weights map. We take universe 0
of `TunedCentralValue_UBGenie` (= Pandora `weightTune`) and both universes of `XSecShape_CCMEC_UBGenie`, and attach
WC's `weight_cv`, `weight_spline`, the sanitized base weight, the open-data net weight, and a few truth variables from
`all_df.parquet`. The join is cached to parquet because both inputs are large.


In [ ]:
cache = f"{CACHE_DIR}/joined_tune_weights.parquet"
if os.path.exists(cache):
    joined = pl.read_parquet(cache)
else:
    spline = pl.scan_parquet(f"{IL}/spline_weights_df.parquet").select(
        JOIN_KEYS + ["filetype",
                     pl.col("TunedCentralValue_UBGenie").list.first().alias("pandora_tune"),
                     pl.col("XSecShape_CCMEC_UBGenie").list.first().alias("xsecshape_u0"),
                     pl.col("XSecShape_CCMEC_UBGenie").list.last().alias("xsecshape_u1"),
                     pl.col("MaCCQE_UBGenie").list.get(3).alias("maccqe_u3"),      # knob value 0 (7-point knob)
                     pl.col("MaCCQE_UBGenie").list.get(6).alias("maccqe_u6")])     # knob value +3 sigma
    alldf = pl.scan_parquet(f"{IL}/all_df.parquet").select(
        JOIN_KEYS + ["detailed_run_period", "wc_weight_cv", "wc_weight_spline", "weight_cv_weight_spline",
                     "wc_net_weight_open_data", "wc_kine_reco_Enu", "wc_truth_nuEnergy", "wc_truth_nuPdg",
                     "wc_truth_nuScatType"])
    joined = spline.join(alldf, on=JOIN_KEYS, how="inner").collect(engine="streaming")
    joined.write_parquet(cache)

# The derived numuCC_rad_corrected / NC_coherent_1g_reweighted rows have no GENIE weights map at all: they were
# appended to spline_weights_df with unit placeholder lists and carry the -1 sentinel in wc_weight_cv, so they are not
# a tune-weight comparison and are left out here.
joined = joined.filter(~pl.col("filetype").is_in(["numuCC_rad_corrected", "NC_coherent_1g_reweighted"]))
joined = joined.with_columns(
    (pl.col("pandora_tune") / pl.col("wc_weight_cv")).alias("tune_ratio"),
).with_columns(
    ((pl.col("tune_ratio") - 1).abs() > MISMATCH_TOL).alias("mismatch"),
)
print(f"{joined.height:,} events with GENIE universes (spline parquet) joined to all_df")


In [ ]:
summary = (
    joined.group_by("filetype").agg([
        pl.len().alias("n_events"),
        pl.col("mismatch").sum().alias("n_mismatch_5pct"),
        pl.col("mismatch").mean().alias("frac_mismatch_5pct"),
        ((pl.col("tune_ratio") - 1).abs() > 0.5).sum().alias("n_mismatch_50pct"),
        ((pl.col("pandora_tune") < 1e-3) & (pl.col("wc_weight_cv") > 1e-2)).sum().alias("n_pandora_tiny_wc_normal"),
        ((pl.col("wc_weight_cv") < 1e-3) & (pl.col("pandora_tune") > 1e-2)).sum().alias("n_wc_tiny_pandora_normal"),
    ])
    .sort("filetype")
)
with pl.Config(tbl_width_chars=200, float_precision=4):
    print(summary)


Half of all **dirt** events disagree, a few thousand `nu_overlay` events, a handful of `nue_overlay`, and none of the
dedicated pi0 samples (generated one interaction per event).

The dangerous class for PROfit is `n_pandora_tiny_wc_normal`: the Pandora tune weight is essentially zero while WC's is
normal, so the event enters the prediction with a normal net weight but with universe weights ~0. Section 3 shows that
in these cases it is the *Pandora* value that belongs to the reported neutrino.


## 2. What the disagreement looks like

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2))
bins = np.linspace(-7, 3, 101)
for ft in FILETYPE_ORDER:
    sub = joined.filter(pl.col("filetype") == ft)
    if sub.height == 0:
        continue
    r = sub["tune_ratio"].to_numpy()
    r = r[np.isfinite(r) & (r > 0)]
    axs[0].hist(np.log10(r), bins=bins, histtype="step", lw=1.5, color=FILETYPE_COLORS[ft], label=f"{ft} ({sub.height:,})")
axs[0].set_yscale("log")
axs[0].set_xlabel(r"$\log_{10}$( Pandora weightTune / WC weight_cv )")
axs[0].set_ylabel("events")
axs[0].set_title("Ratio of the two tune weights, per event")
axs[0].axvspan(np.log10(1 - MISMATCH_TOL), np.log10(1 + MISMATCH_TOL), color="0.85", zorder=0, label=f"agree within {MISMATCH_TOL:.0%}")
axs[0].legend(fontsize=8)

dirt = joined.filter(pl.col("filetype") == "dirt_overlay")
x = np.clip(dirt["wc_weight_cv"].to_numpy(), 1e-7, 50)
y = np.clip(dirt["pandora_tune"].to_numpy(), 1e-7, 50)
h = axs[1].hist2d(np.log10(x), np.log10(y), bins=[np.linspace(-7, 1.7, 88)] * 2, cmap="Blues", norm=mpl.colors.LogNorm())
axs[1].plot([-7, 1.7], [-7, 1.7], "k--", lw=1, label="equal")
axs[1].set_xlabel(r"$\log_{10}$ WC weight_cv")
axs[1].set_ylabel(r"$\log_{10}$ Pandora weightTune (universe 0)")
axs[1].set_title("dirt_overlay: the two weights event by event")
axs[1].legend(loc="upper left")
fig.colorbar(h[3], ax=axs[1], label="events")
plt.tight_layout()
plt.show()


On the left, matched events pile up in the grey band; the mismatched dirt population is spread over many orders of
magnitude. On the right, everything off the diagonal is the mismatched half of dirt: the two weights are simply unrelated
for those events, as expected if they describe two different interactions. The vertical stripe (WC weight normal,
Pandora weight anywhere down to $10^{-7}$) is the population that matters for PROfit: the event enters the prediction
with WC's normal weight although the reported neutrino is an interaction the tune essentially vetoes (Section 3 shows the
Pandora value is the consistent one). The horizontal stripe is the mirror case.


## 3. Root cause in the ntuples

Go back to the checkout files and read, per event:

* WC `T_eval`: `weight_cv`, `weight_spline`, `truth_nuEnergy`, `truth_isCC`; WC `T_PFeval`: `truth_nuScatType`
* Pandora `NeutrinoSelectionFilter`: `nu_e` (the neutrino Pandora reports), `weightTune`, `weightSpline`, and the
  truth particles matched to its reconstructed slice (`mc_pdg`, `mc_purity`)
* gLEE `vertex_tree`: `mctruth_num` (the number of GENIE interactions in the event) and `mctruth_nu_E`
* gLEE `eventweight_tree`: `MCTruth_neutrino_target`

Files: the Run 5 dirt checkout, the Run 4c `nu_overlay` checkout, and (because Runs 4-5 have `weight_spline == 1` for
every event, so only Runs 1-3 can test the spline weight) the Runs 1-3 `nu_overlay` `hist_1` checkout.


In [ ]:
def read_ntuple_weights(filename):
    f = uproot.open(f"{DL}/{filename}")
    ev = f["wcpselection/T_eval"].arrays(["run", "subrun", "event", "weight_cv", "weight_spline", "truth_nuEnergy", "truth_nuPdg", "truth_isCC"], library="np")
    pf = f["wcpselection/T_PFeval"].arrays(["run", "subrun", "event", "truth_nuScatType"], library="np")
    pa = f["nuselection/NeutrinoSelectionFilter"].arrays(["run", "sub", "evt", "nu_e", "nu_pdg", "weightTune", "weightSpline", "mc_pdg", "mc_purity"], library="ak")
    vt = f["singlephotonana/vertex_tree"].arrays(["run_number", "subrun_number", "event_number", "mctruth_num", "mctruth_nu_E"], library="np")
    ew = f["singlephotonana/eventweight_tree"].arrays(["run", "subrun", "event", "MCTruth_neutrino_target"], library="np")
    E = pl.DataFrame({"run": ev["run"], "subrun": ev["subrun"], "event": ev["event"], "wc_weight_cv": ev["weight_cv"],
                      "wc_weight_spline": ev["weight_spline"], "wc_truth_E_MeV": ev["truth_nuEnergy"], "wc_truth_pdg": ev["truth_nuPdg"], "wc_isCC": ev["truth_isCC"]})
    S = pl.DataFrame({"run": pf["run"], "subrun": pf["subrun"], "event": pf["event"], "wc_scat": pf["truth_nuScatType"]})
    slice_is_muon = ak.to_numpy(ak.any((abs(pa["mc_pdg"]) == 13) & (pa["mc_purity"] > 0.5), axis=1))
    P = pl.DataFrame({"run": ak.to_numpy(pa["run"]), "subrun": ak.to_numpy(pa["sub"]), "event": ak.to_numpy(pa["evt"]),
                      "pandora_nu_E_MeV": ak.to_numpy(pa["nu_e"]) * 1000.0, "pandora_nu_pdg": ak.to_numpy(pa["nu_pdg"]),
                      "pandora_weightTune": ak.to_numpy(pa["weightTune"]), "pandora_weightSpline": ak.to_numpy(pa["weightSpline"]),
                      "slice_has_truth_match": ak.to_numpy(ak.num(pa["mc_pdg"]) > 0), "slice_is_muon": slice_is_muon})
    V = pl.DataFrame({"run": vt["run_number"], "subrun": vt["subrun_number"], "event": vt["event_number"],
                      "mctruth_num": vt["mctruth_num"], "glee_nu_E_MeV": vt["mctruth_nu_E"] * 1000.0})
    W = pl.DataFrame({"run": ew["run"], "subrun": ew["subrun"], "event": ew["event"], "target_pdg": ew["MCTruth_neutrino_target"]})
    df = E.join(S, on=["run", "subrun", "event"]).join(P, on=["run", "subrun", "event"]).join(V, on=["run", "subrun", "event"]).join(W, on=["run", "subrun", "event"])
    return df.with_columns([
        ((pl.col("pandora_weightTune") / pl.col("wc_weight_cv") - 1).abs() > MISMATCH_TOL).alias("tune_mismatch"),
        ((pl.col("pandora_weightSpline") / pl.col("wc_weight_spline") - 1).abs() > MISMATCH_TOL).alias("spline_mismatch"),
        ((pl.col("pandora_nu_E_MeV") - pl.col("wc_truth_E_MeV")).abs() > 1.0).alias("pandora_E_ne_wc_E"),
        ((pl.col("glee_nu_E_MeV") - pl.col("wc_truth_E_MeV")).abs() > 1.0).alias("glee_E_ne_wc_E"),
        (pl.col("mctruth_num") > 1).alias("multi"),
        pl.when(pl.col("wc_isCC")).then(pl.lit("CC")).otherwise(pl.lit("NC")).alias("cc"),
        pl.col("wc_scat").replace_strict(SCAT_NAMES, default="other").alias("mode"),
    ])

NTUPLES = {
    "dirt_overlay, run 5":       "checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_dirt_overlay_retuple_retuple_hist_5.root",
    "nu_overlay, run 4c":        "checkout_MCC9.10_Run4acd5_v10_04_07_20_BNB_nu_overlay_retuple_retuple_hist_4c.root",
    "nu_overlay, runs 1-3 (hist_1)": "checkout_MCC9.10_Run123_v10_04_07_20_BNB_nu_overlay_surprise_reco2_hist_1.root",
}
ntuple_dfs = {}
for label, fn in NTUPLES.items():
    cache = f"{CACHE_DIR}/{fn.replace('.root', '')}_tune_check_v2.parquet"
    if os.path.exists(cache):
        ntuple_dfs[label] = pl.read_parquet(cache)
    else:
        ntuple_dfs[label] = read_ntuple_weights(fn)
        ntuple_dfs[label].write_parquet(cache)
    df = ntuple_dfs[label]
    print(f"\n=== {label}: {df.height:,} events (all T_eval entries, not just preselected)")
    print(f"  Pandora nu_e == WC truth_nuEnergy for {(~df['pandora_E_ne_wc_E']).mean():.4%} of events")
    print(f"  gLEE mctruth_nu_E == WC truth_nuEnergy for {(~df['glee_E_ne_wc_E']).mean():.4%} of events")
    print(f"  Pandora weightTune != WC weight_cv (>{MISMATCH_TOL:.0%}) for {df['tune_mismatch'].mean():.2%} of events")
    print(f"  Pandora weightSpline != WC weight_spline for {df['spline_mismatch'].mean():.2%} of events")
    with pl.Config(tbl_rows=12, float_precision=3):
        print(df.group_by("multi").agg([pl.len().alias("n_events"), pl.col("tune_mismatch").mean().alias("frac_tune_mismatch"),
                                        pl.col("spline_mismatch").mean().alias("frac_spline_mismatch"),
                                        (pl.col("wc_weight_spline") != 1).mean().alias("frac_wc_spline_ne_1")]).sort("multi"))


### 3a. The disagreement appears only in events with more than one GENIE interaction

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (label, df) in zip(axs, ntuple_dfs.items()):
    g = (df.group_by("mctruth_num").agg([pl.len().alias("n"), pl.col("tune_mismatch").mean().alias("frac")])
           .filter(pl.col("mctruth_num") <= 8).sort("mctruth_num"))
    n, frac, k = g["n"].to_numpy(), g["frac"].to_numpy(), g["mctruth_num"].to_numpy()
    err = np.sqrt(np.clip(frac * (1 - frac), 0, None) / np.maximum(n, 1))
    color = FILETYPE_COLORS["dirt_overlay" if "dirt" in label else "nu_overlay"]
    ax.errorbar(k, frac, yerr=err, fmt="o-", color=color, lw=1.5, ms=6)
    for xi, yi, ni in zip(k, frac, n):
        ax.annotate(f"{ni:,}", (xi, yi), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8, color="0.3")
    ax.set_xlabel("GENIE interactions in the event (gLEE mctruth_num)")
    ax.set_ylabel("fraction with weightTune != weight_cv")
    ax.set_title(label)
    ax.set_ylim(-0.03, 0.85)
    ax.set_xticks(k)
    ax.axhline(0, color="0.7", lw=0.8)
plt.tight_layout()
plt.show()


Events with exactly one GENIE interaction essentially never disagree (the 0.5% in the Runs 1-3 file are events whose
Pandora `weightTune` is 0, infinite or negative — the known pathological GENIE weights, which WC sanitizes to 1.0; any use
of the Pandora value must keep that sanitization). Events with two or more interactions disagree ~60% of the time,
independent of how many there are. Dirt is dominated by multi-interaction events (a spill has many out-of-cryostat
interactions), which is why half the dirt sample is affected while only ~0.3% of `nu_overlay` is.

Meanwhile Pandora's `nu_e` and gLEE's `mctruth_nu_E` always agree with WC's `truth_nuEnergy`: all frameworks agree on
*which* neutrino the event is about. The question is which of the two weights belongs to it.


### 3b. Which weight belongs to the reported neutrino? A physics-consistency test

The MicroBooNE tune reweights only CC QE and CC MEC (`MaCCQE`, `RPA_CCQE`, `NormCCMEC`, `XSecShape_CCMEC`). So the tune
weight of an NC, RES, DIS or COH event must be exactly 1.0, and the tune weight of a CC QE / CC MEC event is essentially
never exactly 1.0. Single-interaction events (where both weights agree) confirm this rule. Apply it to multi-interaction
events, using WC's own `truth_isCC` / `truth_nuScatType` for the reported neutrino:


In [ ]:
rows = []
for label, df in ntuple_dfs.items():
    t = (df.with_columns([(pl.col("wc_weight_cv") != 1.0).alias("wc_ne_1"), (pl.col("pandora_weightTune") != 1.0).alias("pa_ne_1")])
           .group_by(["multi", "cc", "mode"])
           .agg([pl.len().alias("n"), pl.col("wc_ne_1").mean().alias("frac_WC_weight_cv_ne_1"), pl.col("pa_ne_1").mean().alias("frac_Pandora_weightTune_ne_1")])
           .filter((pl.col("n") >= 100) & pl.col("mode").is_in(["QE", "MEC", "RES", "DIS", "COH"]))
           .with_columns(pl.lit(label).alias("file"))
           .sort(["multi", "cc", "mode"]))
    rows.append(t)
consistency = pl.concat(rows).select(["file", "multi", "cc", "mode", "n", "frac_WC_weight_cv_ne_1", "frac_Pandora_weightTune_ne_1"])
consistency = consistency.with_columns(
    pl.when((pl.col("cc") == "CC") & pl.col("mode").is_in(["QE", "MEC"])).then(pl.lit("!= 1")).otherwise(pl.lit("== 1")).alias("tune_must_be"))
with pl.Config(tbl_rows=60, tbl_width_chars=200, float_precision=3):
    print(consistency)


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.4), sharey=True)
for ax, (label, df) in zip(axs, ntuple_dfs.items()):
    sub = consistency.filter((pl.col("file") == label) & pl.col("multi"))
    cats = [f"{c} {m}" for c, m in zip(sub["cc"], sub["mode"])]
    xpos = np.arange(len(cats))
    ax.bar(xpos - 0.2, sub["frac_WC_weight_cv_ne_1"].to_numpy(), 0.4, color=WC_COLOR, label="WC weight_cv != 1")
    ax.bar(xpos + 0.2, sub["frac_Pandora_weightTune_ne_1"].to_numpy(), 0.4, color=PANDORA_COLOR, label="Pandora weightTune != 1")
    for i, (c, m) in enumerate(zip(sub["cc"], sub["mode"])):
        expected = 1.0 if (c == "CC" and m in ("QE", "MEC")) else 0.0
        ax.plot([i - 0.45, i + 0.45], [expected, expected], color="k", lw=1.5, ls=":", label="expected for the reported neutrino" if i == 0 else None)
    ax.set_xticks(xpos); ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{label}\nevents with > 1 GENIE interaction")
    ax.set_ylim(0, 1.08)
axs[0].set_ylabel("fraction of events with tune weight != 1")
axs[0].legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()


In multi-interaction events **Pandora's `weightTune` follows the rule exactly**: non-unity for essentially every CC QE
and CC MEC event, unity for every NC / RES / DIS / COH event, just like single-interaction events. **WC's `weight_cv` is
non-unity ~40-45% of the time in every category**, i.e. it is uncorrelated with the physics of the neutrino WC itself
reports. It is WC's `weight_cv` that comes from a different `MCTruth`; and in the Runs 1-3 file WC's `weight_spline`
disagrees with Pandora's for 14% of multi-interaction events as well (the spline weight depends on the neutrino energy
and flavour, so it inherits the same mix-up wherever it is not trivially 1).


### 3c. Does WC's `weight_cv` at least belong to the interaction that produced the reconstructed activity?

If WC picked the weight of the interaction truth-matched to its reconstructed neutrino candidate, the wrong-looking
weights would still be the physically *useful* ones. Test: in multi-interaction dirt events whose reported neutrino is NC
(tune must be 1), compare the fraction of events whose Pandora slice is truth-matched to a **muon** (i.e. the detector
activity came from a CC interaction elsewhere in the spill) between events with `weight_cv == 1` and `weight_cv != 1`.


In [ ]:
df = ntuple_dfs["dirt_overlay, run 5"].with_columns((pl.col("wc_weight_cv") != 1.0).alias("wc_ne_1"))
sub = df.filter(pl.col("multi") & (pl.col("cc") == "NC") & pl.col("slice_has_truth_match"))
with pl.Config(float_precision=3):
    print("multi-interaction dirt events whose reported neutrino is NC and whose Pandora slice has a truth match:", sub.height)
    print(sub.group_by("wc_ne_1").agg([pl.len().alias("n"), pl.col("slice_is_muon").mean().alias("frac_slice_matched_to_muon")]).sort("wc_ne_1"))
    ctrl = df.filter(~pl.col("multi") & (pl.col("cc") == "NC") & pl.col("slice_has_truth_match"))
    print("control, single-interaction NC events:")
    print(ctrl.group_by("wc_ne_1").agg([pl.len().alias("n"), pl.col("slice_is_muon").mean().alias("frac_slice_matched_to_muon")]).sort("wc_ne_1"))


Only a weak correlation (25% vs 18%): WC's `weight_cv` does **not** track the interaction that made the reconstructed
activity either. It is best described as the weight of some other `MCTruth` in the event, essentially at random with
respect to both the reported neutrino and the reconstructed one.

Finally, the parquet faithfully carries the Pandora value:


In [ ]:
fn = NTUPLES["dirt_overlay, run 5"]
sp_file = (pl.scan_parquet(f"{IL}/spline_weights_df.parquet").filter(pl.col("filename") == fn)
             .select(["run", "subrun", "event", pl.col("TunedCentralValue_UBGenie").list.first().alias("parquet_u0")])
             .collect(engine="streaming"))
chk = ntuple_dfs["dirt_overlay, run 5"].join(sp_file, on=["run", "subrun", "event"], how="inner")
print(f"{chk.height:,} preselected events of this file are in spline_weights_df; "
      f"parquet universe 0 == Pandora weightTune for {((chk['parquet_u0'] - chk['pandora_weightTune']).abs() < 1e-4).mean():.4%} of them")


## 4. An example: a dirt event with 8 GENIE interactions

Run 25248, subrun 122, event 6139 of the Run 5 dirt checkout has `mctruth_num = 8`, passes the generic preselection
(it is in `spline_weights_df`), and its two tune weights disagree. First, what every tree says about the neutrino it
reports:


In [ ]:
EX = dict(run=25248, subrun=122, event=6139, filename=NTUPLES["dirt_overlay, run 5"])
f = uproot.open(f"{DL}/{EX['filename']}")
vt_rse = f["singlephotonana/vertex_tree"].arrays(["run_number", "subrun_number", "event_number"], library="np")
IDX = int(np.where((vt_rse["run_number"] == EX["run"]) & (vt_rse["subrun_number"] == EX["subrun"]) & (vt_rse["event_number"] == EX["event"]))[0][0])
one = dict(entry_start=IDX, entry_stop=IDX + 1, library="np")

def first(arr):
    v = arr[0]
    return np.asarray(v) if np.ndim(v) else v.item() if hasattr(v, "item") else v

t_eval = {k: first(v) for k, v in f["wcpselection/T_eval"].arrays(["run", "subrun", "event", "truth_nuEnergy", "truth_nuPdg", "truth_isCC", "truth_vtxX", "truth_vtxY", "truth_vtxZ", "weight_cv", "weight_spline", "match_completeness_energy", "match_completeness", "match_purity"], **one).items()}
assert (t_eval["run"], t_eval["subrun"], t_eval["event"]) == (EX["run"], EX["subrun"], EX["event"]), "tree entries are not aligned"
t_pf = {k: first(v) for k, v in f["wcpselection/T_PFeval"].arrays(["truth_nuScatType", "truth_nuIntType"], **one).items()}
pand = {k: first(v) for k, v in f["nuselection/NeutrinoSelectionFilter"].arrays(["nu_e", "nu_pdg", "ccnc", "interaction", "nu_parent_pdg", "true_nu_vtx_x", "true_nu_vtx_y", "true_nu_vtx_z", "weightTune", "weightSpline", "reco_nu_vtx_x", "reco_nu_vtx_y", "reco_nu_vtx_z", "topological_score", "mc_pdg", "mc_E", "mc_vx", "mc_vy", "mc_vz", "mc_purity", "mc_completeness", "all_mc_pdg", "all_mc_E", "all_mc_vx", "all_mc_vy", "all_mc_vz", "all_mc_endx", "all_mc_endy", "all_mc_endz", "mc_generator_pdg", "mc_generator_E", "mc_generator_statuscode"], **one).items()}
glee = {k: first(v) for k, v in f["singlephotonana/vertex_tree"].arrays(["mctruth_num", "mctruth_nu_pdg", "mctruth_nu_E", "mctruth_nu_vertex_x", "mctruth_nu_vertex_y", "mctruth_nu_vertex_z", "mctruth_mode", "mctruth_cc_or_nc", "mctruth_interaction_type", "mctruth_lepton_pdg", "mctruth_lepton_E"], **one).items()}
ewt = {k: first(v) for k, v in f["singlephotonana/eventweight_tree"].arrays(["MCFlux_NuMomE", "MCFlux_ntype", "MCFlux_ptype", "MCTruth_neutrino_CCNC", "MCTruth_neutrino_mode", "MCTruth_neutrino_interactionType", "MCTruth_neutrino_target", "GTruth_tgtPDG"], **one).items()}
lant = {k: first(v) for k, v in f["lantern/EventTree"].arrays(["trueNuE", "trueNuPDG", "trueNuCCNC", "trueNuMode", "trueNuIntrxnType", "trueVtxX", "trueVtxY", "trueVtxZ"], **one).items()}
g4 = {k: first(v) for k, v in f["singlephotonana/geant4_tree"].arrays(["geant4_pdg", "geant4_trackid", "geant4_mother", "geant4_E", "geant4_vx", "geant4_vy", "geant4_vz", "geant4_process", "geant4_end_process"], **one).items()}

ccnc = lambda v: "NC" if v == 1 else "CC"
report = pl.DataFrame({
    "tree":        ["WC T_eval / T_PFeval", "Pandora NeutrinoSelectionFilter", "gLEE vertex_tree", "gLEE eventweight_tree", "LANTERN EventTree"],
    "E_nu [GeV]":  [t_eval["truth_nuEnergy"] / 1000, pand["nu_e"], glee["mctruth_nu_E"], ewt["MCFlux_NuMomE"], lant["trueNuE"]],
    "flavour pdg": [t_eval["truth_nuPdg"], pand["nu_pdg"], glee["mctruth_nu_pdg"], ewt["MCFlux_ntype"], lant["trueNuPDG"]],
    "CC/NC":       [ccnc(0 if t_eval["truth_isCC"] else 1), ccnc(pand["ccnc"]), ccnc(glee["mctruth_cc_or_nc"]), ccnc(ewt["MCTruth_neutrino_CCNC"]), ccnc(lant["trueNuCCNC"])],
    "mode / scat": [f"ScatType {t_pf['truth_nuScatType']} ({SCAT_NAMES.get(int(t_pf['truth_nuScatType']), '?')}), IntType {t_pf['truth_nuIntType']}",
                    f"interaction {pand['interaction']}", f"mode {glee['mctruth_mode']}, type {glee['mctruth_interaction_type']}",
                    f"mode {ewt['MCTruth_neutrino_mode']}, type {ewt['MCTruth_neutrino_interactionType']}", f"mode {lant['trueNuMode']}, type {lant['trueNuIntrxnType']}"],
    "vertex (x, y, z) [cm]": [f"({t_eval['truth_vtxX']:.0f}, {t_eval['truth_vtxY']:.0f}, {t_eval['truth_vtxZ']:.0f})", f"({pand['true_nu_vtx_x']:.0f}, {pand['true_nu_vtx_y']:.0f}, {pand['true_nu_vtx_z']:.0f})",
                              f"({glee['mctruth_nu_vertex_x']:.0f}, {glee['mctruth_nu_vertex_y']:.0f}, {glee['mctruth_nu_vertex_z']:.0f})", "-", f"({lant['trueVtxX']:.0f}, {lant['trueVtxY']:.0f}, {lant['trueVtxZ']:.0f})"],
    "target / parent": ["-", f"parent pdg {pand['nu_parent_pdg']}", "-", f"target {ewt['MCTruth_neutrino_target']}, parent {ewt['MCFlux_ptype']}", "-"],
    "tune weight": [t_eval["weight_cv"], pand["weightTune"], None, None, None],
    "spline weight": [t_eval["weight_spline"], pand["weightSpline"], None, None, None],
})
with pl.Config(tbl_width_chars=250, tbl_rows=10, fmt_str_lengths=60, float_precision=4):
    print(f"run {EX['run']} subrun {EX['subrun']} event {EX['event']}: gLEE mctruth_num = {glee['mctruth_num']}")
    print(report)


All five trees report the same neutrino: a 3.24 GeV $\nu_\mu$ from a $K^+$ decay, scattering **neutral-current
elastic** (GENIE interaction type 1002, `truth_nuScatType` 1 = QE) off a **silicon-28** nucleus (target 1000140280) at
$(x, y, z) = (-640, 304, 2621)$ cm — 26 m downstream of the TPC, in the dirt. The tune does not touch NC elastic at all,
so its tune weight must be exactly 1.0. Pandora's `weightTune` is 1.0. WC's `weight_cv` is 1.173, a typical CC QE tune
weight — it belongs to one of the other seven interactions.

Its GENIE universes are consistent with Pandora's value and with the reported neutrino: every tune-related knob
(`MaCCQE`, `RPA_CCQE`, the MEC knobs) and every other CC / resonance / DIS knob is exactly 1.0 in every universe, and the
only knobs that move are the ones that act on a neutral-current elastic scatter — the NC-elastic form-factor knobs and
the outgoing-nucleon FSI knobs — while the flux universes vary as expected for a 3.2 GeV kaon-parent neutrino.


In [ ]:
row = (pl.scan_parquet(f"{IL}/spline_weights_df.parquet")
         .filter((pl.col("filename") == EX["filename"]) & (pl.col("run") == EX["run"]) & (pl.col("subrun") == EX["subrun"]) & (pl.col("event") == EX["event"]))
         .collect(engine="streaming"))
knob_cols = [c for c, t in row.schema.items() if isinstance(t, pl.List)]
genie = [c for c in knob_cols if c.endswith("_UBGenie")]
varying = [c for c in genie if not np.all(np.asarray(row[c][0], dtype=float) == 1.0)]
print(f"{len(genie) - len(varying)}/{len(genie)} GENIE knobs are exactly 1.0 in every universe; the knobs that vary are:")
for c in varying:
    print(f"  {c:<30} {np.array2string(np.asarray(row[c][0], dtype=float), precision=4, separator=', ')}")
print("selected knobs:")
for c in ["TunedCentralValue_UBGenie", "MaCCQE_UBGenie", "XSecShape_CCMEC_UBGenie", "NormCCMEC_UBGenie", "expskin_FluxUnisim", "kplus_PrimaryHadronFeynmanScaling", "weightsReint"]:
    u = np.asarray(row[c][0], dtype=float)
    print(f"  {c:<36} n_univ={len(u):>4}  first values {np.array2string(u[:5], precision=4, separator=', ')}")


### What about the other seven interactions?

The checkout files keep the GENIE record of **one** `MCTruth` per event in every tree (WC, Pandora, gLEE, LANTERN all
store scalars for a single neutrino, and gLEE's `mctruth_daughters_*` / Pandora's `mc_generator_*` are the particle list
of that same one). The energies, flavours, interaction types and vertices of the remaining interactions are therefore
**not recoverable from these files**. What *is* stored is the Geant4 record of the particles that reached the detector
region — Pandora's `all_mc_*` (the MCParticles kept for the event) and gLEE's `geant4_tree` — which lets us see what
actually produced the reconstructed activity, and infer at least one more interaction:


In [ ]:
pdg, E = np.asarray(pand["all_mc_pdg"]), np.asarray(pand["all_mc_E"])
vx, vy, vz = (np.asarray(pand[k]) for k in ("all_mc_vx", "all_mc_vy", "all_mc_vz"))
ex, ey, ez = (np.asarray(pand[k]) for k in ("all_mc_endx", "all_mc_endy", "all_mc_endz"))
print("MCParticles kept for the event (Pandora all_mc_*):")
for i in range(len(pdg)):
    print(f"  pdg {pdg[i]:>11}  E = {E[i]:7.3f} GeV  start ({vx[i]:7.1f}, {vy[i]:7.1f}, {vz[i]:7.1f})  end ({ex[i]:9.1f}, {ey[i]:9.1f}, {ez[i]:8.1f})")
print("\nGeant4 particles in the gLEE geant4_tree (mother 0 = GENIE final-state particle):")
for i in range(len(g4["geant4_pdg"])):
    print(f"  pdg {g4['geant4_pdg'][i]:>4} trackid {g4['geant4_trackid'][i]:>4} mother {g4['geant4_mother'][i]:>4}  E = {g4['geant4_E'][i]:6.3f} GeV  "
          f"start ({g4['geant4_vx'][i]:7.1f}, {g4['geant4_vy'][i]:7.1f}, {g4['geant4_vz'][i]:7.1f})  process {g4['geant4_process'][i]} -> {g4['geant4_end_process'][i]}")
print(f"\nPandora slice: reco vertex ({pand['reco_nu_vtx_x']:.1f}, {pand['reco_nu_vtx_y']:.1f}, {pand['reco_nu_vtx_z']:.1f}), topological score {pand['topological_score']:.2f}; "
      f"truth-matched to pdg {np.asarray(pand['mc_pdg']).tolist()} with purity {np.round(np.asarray(pand['mc_purity']), 3).tolist()}, completeness {np.round(np.asarray(pand['mc_completeness']), 3).tolist()}")
ex_all = joined.filter((pl.col("run") == EX["run"]) & (pl.col("subrun") == EX["subrun"]) & (pl.col("event") == EX["event"]) & (pl.col("filetype") == "dirt_overlay"))
print(f"WC: reco E_nu = {ex_all['wc_kine_reco_Enu'][0]:.0f} MeV, matched completeness energy {t_eval['match_completeness_energy']:.0f} MeV, "
      f"open-data net weight {ex_all['wc_net_weight_open_data'][0]:.4f} (built from weight_cv = {t_eval['weight_cv']:.3f})")


The only thing in the detector is a **2.53 GeV muon**, a GENIE final-state particle (Geant4 mother 0) born at
$(178, -187, -798)$ cm — 8 m *upstream* of the TPC, in the rock — i.e. the product of a **charged-current $\nu_\mu$
interaction that is one of the other seven**. It enters the TPC, stops at $(93, 3, 141)$ cm and is captured on argon:
the two remaining MCParticles are the capture products, a $^{40}$Cl recoil and a 93 MeV $\nu_\mu$. Pandora's slice is
truth-matched to that muon (purity 0.96), and WC reconstructs 495 MeV from it. The reported 3.24 GeV NC-elastic neutrino
26 m downstream contributed nothing visible. So for this event:

* the *reported* neutrino (all trees): NC elastic on Si, 26 m away — tune weight must be 1 → Pandora is right;
* WC's `weight_cv` = 1.173: the tune weight of some other interaction — possibly the CC one that made the muon, but
  Section 3c shows this is not systematically the case;
* the interaction that actually produced the reconstructed activity is only known through its muon.

Geometry of what we know:


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.6))
tpc = dict(x=(0, 256.35), y=(-116.5, 116.5), z=(0, 1036.8))
mu_start = (g4["geant4_vx"][0], g4["geant4_vy"][0], g4["geant4_vz"][0])
mu_end = (ex[np.abs(pdg) == 13][0], ey[np.abs(pdg) == 13][0], ez[np.abs(pdg) == 13][0])
nu_vtx = (t_eval["truth_vtxX"], t_eval["truth_vtxY"], t_eval["truth_vtxZ"])
reco = (pand["reco_nu_vtx_x"], pand["reco_nu_vtx_y"], pand["reco_nu_vtx_z"])
for ax, (h, hl, hi) in zip(axs, [("x", "x [cm]", 0), ("y", "y [cm]", 1)]):
    ax.add_patch(mpl.patches.Rectangle((tpc["z"][0], tpc[h][0]), tpc["z"][1] - tpc["z"][0], tpc[h][1] - tpc[h][0], fc="0.92", ec="0.4", label="TPC active volume"))
    ax.plot([mu_start[2], mu_end[2]], [mu_start[hi], mu_end[hi]], "-", color=FILETYPE_COLORS["dirt_overlay"], lw=2.5, label="2.53 GeV rock muon (CC interaction elsewhere in the spill)")
    ax.plot(mu_start[2], mu_start[hi], "o", color=FILETYPE_COLORS["dirt_overlay"], ms=8)
    ax.plot(mu_end[2], mu_end[hi], "s", color=FILETYPE_COLORS["dirt_overlay"], ms=7, label="muon stops, captured on Ar")
    ax.plot(nu_vtx[2], nu_vtx[hi], "*", color=PANDORA_COLOR, ms=15, label="reported neutrino: 3.24 GeV NC elastic on Si")
    ax.plot(reco[2], reco[hi], "x", color="k", ms=10, mew=2, label="Pandora reco vertex")
    ax.set_xlabel("z [cm]"); ax.set_ylabel(hl)
    ax.set_xlim(-1000, 2900); ax.grid(alpha=0.3)
axs[0].set_ylim(-800, 450); axs[1].set_ylim(-300, 400)
axs[0].legend(fontsize=8, loc="upper right")
fig.suptitle(f"dirt run {EX['run']} subrun {EX['subrun']} event {EX['event']}: 8 GENIE interactions, one reported, one visible", y=1.02)
plt.tight_layout()
plt.show()


## 5. More examples, and the event behind the multi-pi0 spike

Twelve mismatched dirt events with the smallest Pandora tune weights, and the event that produced the multi-pi0 spike
(run 24506, subrun 240, event 12001).


In [ ]:
examples = (
    joined.filter((pl.col("filetype") == "dirt_overlay") & pl.col("mismatch"))
    .sort("pandora_tune")
    .select(["detailed_run_period", "run", "subrun", "event", "wc_truth_nuPdg", "wc_truth_nuEnergy", "wc_kine_reco_Enu",
             "wc_weight_cv", "pandora_tune", "tune_ratio", "wc_net_weight_open_data", "xsecshape_u1"])
    .head(12)
)
with pl.Config(tbl_width_chars=220, tbl_rows=20, float_precision=5):
    print(examples)
culprit = joined.filter((pl.col("run") == 24506) & (pl.col("subrun") == 240) & (pl.col("event") == 12001) & (pl.col("filetype") == "dirt_overlay"))
with pl.Config(tbl_width_chars=220, float_precision=6):
    print(culprit.select(["filetype", "detailed_run_period", "run", "subrun", "event", "wc_truth_nuScatType", "wc_kine_reco_Enu",
                          "wc_weight_cv", "weight_cv_weight_spline", "pandora_tune", "wc_net_weight_open_data",
                          "xsecshape_u0", "xsecshape_u1"]))


In [ ]:
# every GENIE knob of the culprit event, as stored in spline_weights_df (first 7 universes shown)
row = (pl.scan_parquet(f"{IL}/spline_weights_df.parquet")
         .filter((pl.col("run") == 24506) & (pl.col("subrun") == 240) & (pl.col("event") == 12001) & (pl.col("filetype") == "dirt_overlay"))
         .collect(engine="streaming"))
knob_cols = [c for c, t in row.schema.items() if isinstance(t, pl.List) and c.endswith("_UBGenie")]
print(f"{'knob':<30} {'n_univ':>6}  universes")
for c in knob_cols:
    u = np.asarray(row[c][0], dtype=float)
    print(f"{c:<30} {len(u):>6}  {np.array2string(u[:7], precision=6, separator=', ')}")


The culprit's reported neutrino is a 621 MeV $\nu_\mu$ **CC MEC on oxygen-16** (a 2-interaction event). Its Pandora
tune weight is 4.5e-6 — the tune's MEC reweighting does drive some MEC events to essentially zero — and every tune-scaled
knob sits at that value in every universe; the one exception is `XSecShape_CCMEC`, whose second universe (1.6559) is the
MEC shape-model weight relative to the *untuned* prediction and is not multiplied by the tune. WC's `weight_cv` = 1.38 is
the other interaction's tune weight, so the event was written into the PROfit file with a normal net weight (0.089 after
the test-half upweight) instead of the ~3e-7 the reported neutrino's own weight would give.


## 6. Consequences

### 6a. The CV prediction

`postprocessing.py` builds the base weight as WC `weight_cv * weight_spline`. For the mismatched events that is another
interaction's tune (and, in Runs 1-3, spline) weight. Per filetype this affects the fractions in Section 1: about half
of dirt, 0.4% of `nu_overlay`. The *average* dirt weight is not necessarily far off (the wrong weight is drawn from the
same population), but the per-event weights, and hence the shape, are.

### 6b. Our systematics (`systematics.py`)

The universe histograms divide each stored universe by `wc_weight_cv` per event. For a mismatched event the constant
factor `pandora_tune / wc_weight_cv` appears in **every** universe: the event's contribution is scaled by the same wrong
number everywhere, which the fractional covariance sees as one event moving by $(\text{ratio} - 1)$ in every universe —
for the culprit, -100%. Bounded and tiny in any populated bin, but wrong.


In [ ]:
r = culprit["tune_ratio"][0]
w = culprit["wc_net_weight_open_data"][0]
print(f"culprit: net weight {w:.4f} (from WC weight_cv), pandora_tune/wc_weight_cv = {r:.3g}")
print(f"  -> with the reported neutrino's own tune weight the net weight would be {w * r:.3g}")
print(f"  -> in every GENIE universe the notebooks currently give this event weight {w * r:.3g} instead of {w:.4f}: a {100*(r-1):.1f}% shift of one event")


### 6c. PROfit: `force_0_cv` normalizes per **bin**, so a lone mismatched event blows up a subchannel bin

PROfit builds, for each systematic and each *full* bin (subchannel x energy bin), the ratio of the universe spectrum to
the CV spectrum, and with `force_0_cv="true"` divides all ratios by the ratio at knob value 0
(`PROsyst::FillSpline`). For a bin containing a single event that is
$$\frac{\sum_i w_i u_{k,i}}{\sum_i w_i u_{0,i}} = \frac{u_k}{u_0},$$
independent of the event weight. For tune-scaled knobs $u_k/u_0$ is the knob ratio, but for `XSecShape_CCMEC` universe 1
is not tune-scaled, so $u_1/u_0 = 1.6559 / 4.5\times10^{-6} = 3.7\times10^5$. The *absolute* effect on the channel bin
is that ratio times the event's weight, which is where the wrong CV weight enters: with WC's 0.089 the implied fractional
error of the collapsed multi-pi0 bin is $\sim 4000$; with the reported neutrino's own weight ($\sim 3\times10^{-7}$) it
would be $\sim 0.01$.

Below we reproduce PROfit's bin ratios from the PROfit input file for the two dirt subchannel bins that spiked in the
2026-09-09 outputs (`test_r15_21`): `multipi0_dirt` 700-800 MeV and `other2g_dirt` 900-1200 MeV (bin edges from
`bagels_full_open_data_v03.xml`), to be compared with the ~4000 and ~430 read off PROfit's
`test_r15_21_v1_fractional_systematics.pdf`.


In [ ]:
profit_file = f"{IL}/minimal_withspline_df.root"
# reco_category indices from save_PROfit_rootfiles.RECO_CATEGORIES: 15 = multi_pi0, 16 = eta_other ("other2g")
CASES = [("multipi0", 15, 700.0, 800.0), ("other2g", 16, 900.0, 1200.0)]
KNOBS = ["XSecShape_CCMEC_UBGenie", "DecayAngMEC_UBGenie", "MaCCQE_UBGenie", "NormCCMEC_UBGenie"]
CV_INDEX = {2: 0, 5: 1, 6: 2, 7: 3}   # position of knob value 0 in the universe list, per list length (XML knobvals)

if not os.path.exists(profit_file):
    print(f"{profit_file} not found; skipping the PROfit reproduction")
else:
    tree = uproot.open(profit_file)["tree"]
    rows = []
    for ch, cat, lo, hi in CASES:
        a = tree.arrays(["filetype", "net_weight", "isdirt", "isext", "run", "subrun", "event", "TunedCentralValue_UBGenie"] + KNOBS,
                        cut=f"(reco_category=={cat}) & (wc_kine_reco_Enu>={lo}) & (wc_kine_reco_Enu<{hi}) & (isdata==0)", library="np")
        w = a["net_weight"]
        for sub, m in [("overlays", (a["isdirt"] == 0) & (a["isext"] == 0)), ("dirt", a["isdirt"] == 1)]:
            if m.sum() == 0:
                continue
            cv_sub, cv_ch = w[m].sum(), w[(a["isext"] == 0)].sum()
            for k in KNOBS:
                U = np.array([np.asarray(u, dtype=float) for u in a[k][m]])
                spec = (U * w[m, None]).sum(axis=0)
                i0 = CV_INDEX[U.shape[1]]
                with np.errstate(divide="ignore", invalid="ignore"):
                    rn = spec / spec[i0]
                worst = rn[np.nanargmax(np.abs(rn - 1))]
                rows.append(dict(channel=ch, subchannel=sub, bin=f"{lo:.0f}-{hi:.0f}", n_events=int(m.sum()), cv_subchannel=cv_sub,
                                 cv_channel=cv_ch, knob=k, worst_normalized_ratio=worst,
                                 implied_frac_error_of_channel_bin=abs(worst - 1) * cv_sub / cv_ch))
    tab = pl.DataFrame(rows).sort(["channel", "subchannel", "knob"])
    with pl.Config(tbl_width_chars=220, tbl_rows=40, float_precision=4):
        print(tab)


The `XSecShape_CCMEC` row of each dirt subchannel bin reproduces PROfit's numbers: normalized ratios of
$3.7\times10^5$ (multipi0) and $2.8\times10^4$ (other2g), i.e. implied fractional errors of order $10^3$ on the channel
bin, while every tune-scaled knob in the same bin is benign and the overlays subchannel (hundreds of consistent events)
is unaffected.

Which PROfit outputs show it: not the **fit** (best-fit spectra and $\chi^2$ are identical with and without the legacy
band, 297.1 / 297 bins), but the **prior fractional-systematics plot** (the ~4000 "MEC" entry) and the
**`--legacy-postfit-error` band**, which throws from the *prior* covariance with no data constraint
(`getMCMCErrorBand`, `use_data == false`); the default post-fit band constrains that mode with the data.


### 6d. Two facts about the two-universe knobs (needed to read the PROfit XML correctly)

`XSecShape_CCMEC` universe 1 behaves differently from every other knob. For non-MEC events it is *identical to the
tune weight* (universe 0), i.e. no variation, as it should be. For MEC events it is the constant 1.6559 (or 1.0)
**regardless of the event's tune weight** (median $u_1/\text{tune}$ = 4.3 for MEC), so it is a weight relative to the
**untuned** GENIE MEC prediction rather than `tune x ratio`. That is exactly why the bin-level ratio $u_1/u_0$ explodes
for a MEC event with a tiny tune weight, while every genuinely tune-scaled knob stays at $u_k/u_0 \approx$ knob ratio.
Separately, for four of the six two-universe knobs the CV is universe **1**, not universe 0, so `knobvals="0, 1"` in the
XML labels them backwards.


In [ ]:
nu = joined.filter(pl.col("filetype") == "nu_overlay")
t = (nu.group_by("wc_truth_nuScatType").agg([
        pl.len().alias("n"),
        (pl.col("xsecshape_u1") == 1.0).mean().alias("frac_u1_eq_1"),
        ((pl.col("xsecshape_u1") - 1.655945).abs() < 1e-5).mean().alias("frac_u1_eq_1.6559"),
        pl.col("xsecshape_u1").quantile(0.05).alias("u1_q05"), pl.col("xsecshape_u1").median().alias("u1_median"),
        pl.col("xsecshape_u1").quantile(0.95).alias("u1_q95"),
        (pl.col("xsecshape_u1") / pl.col("pandora_tune")).median().alias("median_u1_over_tune"),
     ]).sort("wc_truth_nuScatType")
     .with_columns(pl.col("wc_truth_nuScatType").replace_strict(SCAT_NAMES, default="other").alias("mode")))
with pl.Config(tbl_width_chars=200, float_precision=3):
    print("XSecShape_CCMEC universe 1 by interaction mode (nu_overlay):")
    print(t.select(["mode", "n", "frac_u1_eq_1", "frac_u1_eq_1.6559", "u1_q05", "u1_median", "u1_q95", "median_u1_over_tune"]))


In [ ]:
two_univ = ["XSecShape_CCMEC_UBGenie", "DecayAngMEC_UBGenie", "AxFFCCQEshape_UBGenie", "VecFFCCQEshape_UBGenie",
            "Theta_Delta2Npi_UBGenie", "ThetaDelta2NRad_UBGenie"]
sample = (pl.scan_parquet(f"{IL}/spline_weights_df.parquet").filter(pl.col("filetype") == "nu_overlay")
            .select([pl.col("TunedCentralValue_UBGenie").list.first().alias("tune")]
                    + [pl.col(k).list.first().alias(k + "_0") for k in two_univ]
                    + [pl.col(k).list.last().alias(k + "_1") for k in two_univ])
            .head(300_000).collect(engine="streaming"))
print(f"{'knob':<28} {'univ0 == tune':>14} {'univ1 == tune':>14}   -> CV is universe")
for k in two_univ:
    f0 = (sample[k + "_0"] == sample["tune"]).mean(); f1 = (sample[k + "_1"] == sample["tune"]).mean()
    cv = "0" if f0 > 0.999 and f1 < 0.999 else ("1" if f1 > 0.999 and f0 < 0.999 else "both (knob has no variation)")
    print(f"{k:<28} {f0:>14.3f} {f1:>14.3f}   -> {cv}")


## 7. Conclusions and recommendations

1. **Root cause is in the ntuples, not in our processing.** In events with more than one GENIE interaction, WC's
   `T_eval` `weight_cv` (and `weight_spline`) is taken from a different `MCTruth` than the neutrino WC's own `truth_nu*`
   branches — and every other tree — report, ~60% of the time. Pandora's `weights` map (`weightTune`, `weightSpline`,
   and all GENIE / flux / reint universes) is consistent with the reported neutrino. This hits ~49% of dirt events,
   ~0.4% of `nu_overlay`, ~0.01% of `nue_overlay`, and none of the dedicated pi0 samples. WC's wrong weight does not
   track the interaction that made the reconstructed activity either (Section 3c).
2. **Our CV prediction is wrong for those events** (Section 6a), our covariance code divides by the wrong number for
   them (6b), and PROfit's per-bin `force_0_cv` turns the worst cases into $10^3$-level fractional errors in sparse
   dirt subchannel bins (6c).
3. **Suggested fix, at the CV-weight level:** read Pandora's `weightTune` and `weightSpline` from
   `nuselection/NeutrinoSelectionFilter` into `all_df` and use them instead of WC `weight_cv` / `weight_spline` in the
   base weight of `postprocessing.py`, keeping the existing sanitization (non-finite, $\le 0$ or $> 30$ → 1.0, which is
   also what WC does to the pathological raw values seen in Section 3a). Then the CV weight, the systematics division
   and the stored universes all describe the same interaction, PROfit's `force_0_cv` becomes exact event by event, and
   the culprit event drops to its proper ~3e-7 weight. Note the remaining convention: every framework weights the
   *reported* neutrino (`MCTruth` index 0), which for pile-up dirt events is not necessarily the interaction that
   produced the visible activity (Section 4).
4. **XML hygiene:** `DecayAngMEC`, `AxFFCCQEshape`, `VecFFCCQEshape` and `Theta_Delta2Npi` have their CV in universe 1,
   so their `knobvals` should be `"1, 0"` (or the universe order swapped when writing); `ThetaDelta2NRad` carries no
   variation at all; `XSecShape_CCMEC` is relative to the untuned MEC prediction for MEC events.
